In [1]:
# Célula 1 — Configuração da SparkSession com suporte ao MinIO
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)
# Endereço do MinIO acessível de dentro do contêiner Docker
# "host.docker.internal" resolve para o IP do host a partir do contêiner
MINIO_ENDPOINT = "http://host.docker.internal:9000"
MINIO_ACCESS_KEY = "minioadmin"

MINIO_SECRET_KEY = "minioadmin123"
# Criar a SparkSession com os pacotes necessários para S3
spark = (
    SparkSession.builder
    .appName("DataEngineeringCourse-Aula3" )
    .master("local[*]")  # Usar todos os núcleos de CPU disponíveis
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    # Configurações do conector S3A para MinIO
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # Otimizações de performance
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.ui.port", "4040")
    .getOrCreate()
    )

# Configurar nível de log para reduzir verbosidade
spark.sparkContext.setLogLevel("WARN")
print(f" ✅ SparkSession criada com sucesso!")
print(f"   Versão do Spark: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   Núcleos disponíveis: {spark.sparkContext.defaultParallelism}")
print(f"\n    Spark UI disponível em: http://localhost:4040" )

 ✅ SparkSession criada com sucesso!
   Versão do Spark: 3.5.0
   Master: local[*]
   Núcleos disponíveis: 28

    Spark UI disponível em: http://localhost:4040


In [2]:
# Outra celula
# Célula 2 — Leitura do dataset de e-commerce da camada Bronze
import time
# Caminho S3A para os arquivos Parquet no MinIO
# O prefixo s3a:// indica o conector Hadoop para S3
CAMINHO_ECOMMERCE = "s3a://bronze/ecommerce_sintetico/"
print("Lendo dataset de e-commerce do MinIO...")
inicio = time.time()
df_pedidos = spark.read.parquet(CAMINHO_ECOMMERCE)
# Nota: a leitura é LAZY — o Spark ainda não processou nenhum dado
# O schema é inferido dos metadados do Parquet (sem ler os dados)
duracao = time.time() - inicio
print(f" ✅ Schema inferido em {duracao:.2f}s (leitura lazy — dados ainda não caregados)")
print(f"\nSchema do DataFrame:")
df_pedidos.printSchema()

Lendo dataset de e-commerce do MinIO...
 ✅ Schema inferido em 2.02s (leitura lazy — dados ainda não caregados)

Schema do DataFrame:
root
 |-- id_pedido: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- id_produto: long (nullable = true)
 |-- categoria: string (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- quantidade: long (nullable = true)
 |-- status_pedido: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- data_pedido: timestamp_ntz (nullable = true)
 |-- avaliacao_cliente: long (nullable = true)
 |-- _fonte: string (nullable = true)
 |-- _ingerido_em: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: double (nullable = true)



In [3]:
# Célula 3 — Primeira ação: contar registros (força a execução do plano)
print("Contando registros (primeira ação — força execução do plano Spark)...")
inicio = time.time()
 
total_registros = df_pedidos.count()
 
duracao = time.time() - inicio
print(f"✅ Total de registros: {total_registros:,}")
print(f"   Tempo de execução: {duracao:.2f}s")
print(f"\n   Acesse o Spark UI em http://localhost:4040 para ver o plano de execução!")

Contando registros (primeira ação — força execução do plano Spark)...
✅ Total de registros: 1,000,000
   Tempo de execução: 0.94s

   Acesse o Spark UI em http://localhost:4040 para ver o plano de execução!


In [4]:
# Célula 3 — Primeira ação: contar registros (força a execução do plano)
print("Contando registros (primeira ação — força execução do plano Spark)...")
inicio = time.time()
 
total_registros = df_pedidos.count()
 
duracao = time.time() - inicio
print(f"✅ Total de registros: {total_registros:,}")
print(f"   Tempo de execução: {duracao:.2f}s")
print(f"\n   Acesse o Spark UI em http://localhost:4040 para ver o plano de execução!")

Contando registros (primeira ação — força execução do plano Spark)...
✅ Total de registros: 1,000,000
   Tempo de execução: 0.20s

   Acesse o Spark UI em http://localhost:4040 para ver o plano de execução!


In [5]:
# Célula 4 — Exploração inicial do DataFrame
# show() é uma ação que força a execução e exibe os dados
print("Amostra dos dados (5 primeiras linhas):")
df_pedidos.show(5, truncate=False)
 
print(f"\nEstatísticas descritivas das colunas numéricas:")
df_pedidos.select(
    "valor_unitario", "quantidade", "valor_total"
).describe().show()


Amostra dos dados (5 primeiras linhas):
+---------+----------+----------+-----------+--------------+----------+-------------+--------+-------------------+-----------------+--------------------+--------------------------------+-----------+----+---+----+
|id_pedido|id_cliente|id_produto|categoria  |valor_unitario|quantidade|status_pedido|regiao  |data_pedido        |avaliacao_cliente|_fonte              |_ingerido_em                    |valor_total|ano |mes|dia |
+---------+----------+----------+-----------+--------------+----------+-------------+--------+-------------------+-----------------+--------------------+--------------------------------+-----------+----+---+----+
|1        |15796     |111       |Alimentos  |973.76        |1         |concluido    |Nordeste|2022-01-01 00:00:00|5                |sistema_ecommerce_v2|2026-06-25T22:33:16.575177+00:00|973.76     |2026|6  |25.0|
|2        |861       |3214      |Esportes   |604.88        |1         |concluido    |Nordeste|2022-01-01 00:

In [6]:
# Célula 5 — Filtros e Seleções (equivalente ao WHERE e SELECT do SQL)
 
# Filtrar apenas pedidos concluidos com valor total acima de R$ 500
df_pedidos_premium = (
    df_pedidos
    .filter(
        (F.col("status_pedido") == "concluido") &
        (F.col("valor_total") > 500.0)
    )
    .select(
        "id_pedido",
        "id_cliente",
        "categoria",
        "valor_total",
        "regiao",
        "data_pedido",
    )
)
 
print("Pedidos concluidos com valor > R$ 500:")
print(f"  Total filtrado: {df_pedidos_premium.count():,} registros")
df_pedidos_premium.show(5)

Pedidos concluidos com valor > R$ 500:
  Total filtrado: 697,588 registros
+---------+----------+-----------+-----------+--------+-------------------+
|id_pedido|id_cliente|  categoria|valor_total|  regiao|        data_pedido|
+---------+----------+-----------+-----------+--------+-------------------+
|        1|     15796|  Alimentos|     973.76|Nordeste|2022-01-01 00:00:00|
|        2|       861|   Esportes|     604.88|Nordeste|2022-01-01 00:00:30|
|        3|     76821|     Livros|    3655.18| Sudeste|2022-01-01 00:01:00|
|        4|     54887|Eletrônicos|     6757.7|Nordeste|2022-01-01 00:01:30|
|        5|      6266|     Roupas|    7931.43|   Norte|2022-01-01 00:02:00|
+---------+----------+-----------+-----------+--------+-------------------+
only showing top 5 rows



In [7]:
# Célula 6 — Agrupamentos e Agregações (equivalente ao GROUP BY do SQL)
# Receita total e ticket médio por categoria e região
df_receita_categoria = (
    df_pedidos
    .filter(F.col("status_pedido") == "concluido")
    .groupBy("categoria", "regiao")
    .agg(
        F.count("id_pedido").alias("total_pedidos"),
        F.sum("valor_total").alias("receita_total"),
        F.avg("valor_total").alias("ticket_medio"),
        F.countDistinct("id_cliente").alias("clientes_unicos"),
    )
    .withColumn("receita_total", F.round("receita_total", 2))
    .withColumn("ticket_medio", F.round("ticket_medio", 2))
    .orderBy(F.desc("receita_total"))
)
print("Receita por Categoria e Região (pedidos concluidos):")
df_receita_categoria.show(15)

Receita por Categoria e Região (pedidos concluidos):
+-----------+------------+-------------+--------------+------------+---------------+
|  categoria|      regiao|total_pedidos| receita_total|ticket_medio|clientes_unicos|
+-----------+------------+-------------+--------------+------------+---------------+
|   Esportes|         Sul|        30293|1.6785525032E8|     5541.06|          26222|
|  Alimentos|         Sul|        30235|1.6755700927E8|     5541.82|          26112|
|     Roupas|    Nordeste|        30273|1.6743142891E8|     5530.72|          26131|
|  Alimentos|    Nordeste|        30238|1.6727397283E8|     5531.91|          26062|
|     Roupas|Centro-Oeste|        30266|1.6704575814E8|     5519.25|          26224|
|   Esportes|     Sudeste|        30075|1.6671991041E8|     5543.47|          25941|
|     Livros|Centro-Oeste|        30090|1.6654848877E8|     5535.01|          25979|
|Eletrônicos|Centro-Oeste|        30089|1.6633670362E8|     5528.16|          25962|
|Eletrônicos

In [8]:
# Célula 7 — Funções de Janela (Window Functions)
# Calcular o ranking de receita por categoria dentro de cada região
from pyspark.sql.window import Window
janela_regiao = Window.partitionBy("regiao").orderBy(F.desc("receita_total"))
df_ranking = (
    df_receita_categoria
    .withColumn("ranking_na_regiao", F.rank().over(janela_regiao))
    .filter(F.col("ranking_na_regiao") <= 3)  # Top 3 categorias por região
    .orderBy("regiao", "ranking_na_regiao")
)
print("Top 3 categorias por receita em cada região:")
df_ranking.show(20)

Top 3 categorias por receita em cada região:
+-----------+------------+-------------+--------------+------------+---------------+-----------------+
|  categoria|      regiao|total_pedidos| receita_total|ticket_medio|clientes_unicos|ranking_na_regiao|
+-----------+------------+-------------+--------------+------------+---------------+-----------------+
|     Roupas|Centro-Oeste|        30266|1.6704575814E8|     5519.25|          26224|                1|
|     Livros|Centro-Oeste|        30090|1.6654848877E8|     5535.01|          25979|                2|
|Eletrônicos|Centro-Oeste|        30089|1.6633670362E8|     5528.16|          25962|                3|
|     Roupas|    Nordeste|        30273|1.6743142891E8|     5530.72|          26131|                1|
|  Alimentos|    Nordeste|        30238|1.6727397283E8|     5531.91|          26062|                2|
|   Esportes|    Nordeste|        29817|1.6440033902E8|     5513.64|          25774|                3|
|     Livros|       Norte|  

In [9]:
# Celula 8
df_enriquecido = (
    df_pedidos
    .withColumn(
        # Extrair o mês da data do pedido
        "mes_pedido", F.month("data_pedido")
    )
    .withColumn(
        # Extrair o ano
        "ano_pedido", F.year("data_pedido")
    )
    .withColumn(
        # Classificar o pedido por faixa de valor
        "faixa_valor",
        F.when(F.col("valor_total") < 100, "Baixo")
         .when(F.col("valor_total") < 500, "Médio")
         .when(F.col("valor_total") < 1500, "Alto")
         .otherwise("Premium")
    )
    .withColumn(
        # Flag para pedidos com avaliação positiva (4 ou 5 estrelas)
        "avaliacao_positiva",
        F.when(F.col("avaliacao_cliente") >= 4, True).otherwise(False)
    )
)

In [10]:
print("DataFrame enriquecido com colunas derivadas:")
df_enriquecido.select(
"id_pedido", "valor_total", "faixa_valor",
"mes_pedido", "ano_pedido", "avaliacao_positiva"
).show(10)

DataFrame enriquecido com colunas derivadas:
+---------+-----------+-----------+----------+----------+------------------+
|id_pedido|valor_total|faixa_valor|mes_pedido|ano_pedido|avaliacao_positiva|
+---------+-----------+-----------+----------+----------+------------------+
|        1|     973.76|       Alto|         1|      2022|              true|
|        2|     604.88|       Alto|         1|      2022|             false|
|        3|    3655.18|    Premium|         1|      2022|              true|
|        4|     6757.7|    Premium|         1|      2022|              true|
|        5|    7931.43|    Premium|         1|      2022|              true|
|        6|     6652.0|    Premium|         1|      2022|              true|
|        7|    4120.02|    Premium|         1|      2022|              true|
|        8|     4277.0|    Premium|         1|      2022|              true|
|        9|    4833.92|    Premium|         1|      2022|              true|
|       10|      13.75|      Ba

In [11]:

# Célula 9 — Visualização do Plano de Execução (Catalyst Optimizer)
# O Spark usa o Catalyst Optimizer para transformar o plano lógico
# em um plano físico otimizado antes de executar qualquer operação.
print("=== Plano de Execução Lógico (não otimizado) ===")
df_receita_categoria.explain(mode="simple")
print("\n=== Plano de Execução Físico (otimizado pelo Catalyst) ===")
df_receita_categoria.explain(mode="cost")

=== Plano de Execução Lógico (não otimizado) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [receita_total#486 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(receita_total#486 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=771]
      +- HashAggregate(keys=[categoria#3, regiao#7], functions=[count(id_pedido#0L), sum(valor_total#12), avg(valor_total#12), count(distinct id_cliente#1L)])
         +- Exchange hashpartitioning(categoria#3, regiao#7, 200), ENSURE_REQUIREMENTS, [plan_id=768]
            +- HashAggregate(keys=[categoria#3, regiao#7], functions=[merge_count(id_pedido#0L), merge_sum(valor_total#12), merge_avg(valor_total#12), partial_count(distinct id_cliente#1L)])
               +- HashAggregate(keys=[categoria#3, regiao#7, id_cliente#1L], functions=[merge_count(id_pedido#0L), merge_sum(valor_total#12), merge_avg(valor_total#12)])
                  +- Exchange hashpartitioning(categoria#3, regiao#7, id_cliente#1L, 200), ENSURE_REQUIREMENTS,

In [12]:
# Célula 10 — Cache de DataFrames para Reutilização
# Quando um DataFrame é usado múltiplas vezes, o cache evita
# que o Spark releia os dados do MinIO a cada operação.
print("Armazenando DataFrame em cache...")
inicio = time.time()
df_pedidos.cache()
df_pedidos.count()  # Força a materialização do cache
duracao_cache = time.time() - inicio
print(f"  Cache materializado em {duracao_cache:.2f}s")
# Segunda contagem — agora vem do cache (muito mais rápida)
inicio = time.time()
df_pedidos.count()
duracao_cache_hit = time.time() - inicio
print(f"  Leitura do cache: {duracao_cache_hit:.4f}s")
print(f"  Speedup: {duracao_cache / duracao_cache_hit:.0f}x mais rápido com cache")

Armazenando DataFrame em cache...
  Cache materializado em 5.60s
  Leitura do cache: 0.0451s
  Speedup: 124x mais rápido com cache


In [13]:
# Célula 10 — Cache de DataFrames para Reutilização
# Quando um DataFrame é usado múltiplas vezes, o cache evita
# que o Spark releia os dados do MinIO a cada operação.
print("Armazenando DataFrame em cache...")
inicio = time.time()
df_pedidos.cache()
df_pedidos.count()  # Força a materialização do cache
duracao_cache = time.time() - inicio
print(f"  Cache materializado em {duracao_cache:.2f}s")
# Segunda contagem — agora vem do cache (muito mais rápida)
inicio = time.time()
df_pedidos.count()
duracao_cache_hit = time.time() - inicio
print(f"  Leitura do cache: {duracao_cache_hit:.4f}s")
print(f"  Speedup: {duracao_cache / duracao_cache_hit:.0f}x mais rápido com cache")

Armazenando DataFrame em cache...
  Cache materializado em 0.04s
  Leitura do cache: 0.0600s
  Speedup: 1x mais rápido com cache


In [14]:
# Célula 10 — Cache de DataFrames para Reutilização
# Quando um DataFrame é usado múltiplas vezes, o cache evita
# que o Spark releia os dados do MinIO a cada operação.
print("Armazenando DataFrame em cache...")
inicio = time.time()
df_pedidos.cache()
df_pedidos.count()  # Força a materialização do cache
duracao_cache = time.time() - inicio
print(f"  Cache materializado em {duracao_cache:.2f}s")
# Segunda contagem — agora vem do cache (muito mais rápida)
inicio = time.time()
df_pedidos.count()
duracao_cache_hit = time.time() - inicio
print(f"  Leitura do cache: {duracao_cache_hit:.4f}s")
print(f"  Speedup: {duracao_cache / duracao_cache_hit:.0f}x mais rápido com cache")

Armazenando DataFrame em cache...
  Cache materializado em 0.05s
  Leitura do cache: 0.0436s
  Speedup: 1x mais rápido com cache


In [15]:
# Célula 11 — Gravação do resultado processado de volta no MinIO
# Os resultados são gravados como Parquet particionado por categoria
 
CAMINHO_SAIDA = "s3a://bronze/ecommerce_processado/"
 
print(f"Gravando resultados em: {CAMINHO_SAIDA}")
inicio = time.time()
 
(
    df_enriquecido
    .filter(F.col("status_pedido") == "concluido")
    .write
    .mode("overwrite")
    .partitionBy("ano_pedido", "mes_pedido")  # Particionamento físico no S3
    .parquet(CAMINHO_SAIDA)
)
 
duracao = time.time() - inicio
print(f"✅ Dados gravados em {duracao:.2f}s")
print(f"   Particionado por: ano_pedido / mes_pedido")
print(f"   Formato: Parquet + Snappy (padrão do Spark)")
 
# Verificar os arquivos criados
df_verificacao = spark.read.parquet(CAMINHO_SAIDA)
print(f"\n   Registros gravados: {df_verificacao.count():,}")
print(f"   Partições físicas criadas:")

Gravando resultados em: s3a://bronze/ecommerce_processado/
✅ Dados gravados em 3.06s
   Particionado por: ano_pedido / mes_pedido
   Formato: Parquet + Snappy (padrão do Spark)

   Registros gravados: 750,572
   Partições físicas criadas:


In [16]:
df_verificacao.select("ano_pedido", "mes_pedido").distinct().orderBy("ano_pedido", "mes_pedido").show(5)

+----------+----------+
|ano_pedido|mes_pedido|
+----------+----------+
|      2022|         1|
|      2022|         2|
|      2022|         3|
|      2022|         4|
|      2022|         5|
+----------+----------+
only showing top 5 rows

